# Step 4: Final Optimized Cross-Task Evaluation

This version removes all manual re-implementations of sliding windows and metric math. It treats the `eomt` repository as a library, calling the same internal methods used by the training CLI.

In [4]:
!pip install lightning jsonargparse

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 48.9 MB/s eta 0:00:00


In [5]:
from google.colab import drive
import os
import sys
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Environment Configuration
drive.mount('/content/drive')
project_root = '/content/drive/MyDrive/FundGitHubProject'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

# We extracted everything, so 'eval' should be in our project_root
try:
    from eval.iouEval import iouEval
except ModuleNotFoundError:
    from iouEval import iouEval

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cpu


In [5]:
import os

eval_root_path = '/content/drive/MyDrive/FundGitHubProject/eval'
if os.path.exists(eval_root_path):
    print(f"Contents of {eval_root_path}:")
    for item in os.listdir(eval_root_path):
        print(item)
else:
    print(f"Directory does not exist: {eval_root_path}")

Directory does not exist: /content/drive/MyDrive/FundGitHubProject/eval


In [7]:
import os
import shutil

source_dir = '/content/drive/MyDrive/FundGitHubProject/eomt'
target_dir = '/content/drive/MyDrive/FundGitHubProject'

if os.path.exists(source_dir):
    print(f"Moving contents from {source_dir} to {target_dir}...")
    for item in os.listdir(source_dir):
        s = os.path.join(source_dir, item)
        d = os.path.join(target_dir, item)
        try:
            shutil.move(s, d)
            print(f"Moved: {item}")
        except Exception as e:
            print(f"Error moving {item}: {e}")
    print("Extraction complete!")
else:
    print(f"Source directory {source_dir} not found.")

Moving contents from /content/drive/MyDrive/FundGitHubProject/eomt to /content/drive/MyDrive/FundGitHubProject...
Moved: .gitignore
Moved: LICENSE
Moved: README.md
Moved: __init__.py
Moved: configs
Moved: datasets
Moved: docs
Moved: inference.ipynb
Moved: main.py
Moved: models
Moved: requirements.txt
Moved: training
Moved: eval
Moved: data
Moved: eomt_coco.bin
Moved: .ipynb_checkpoints
Moved: eomt_cityscapes.bin
Moved: eomt_cityscapes_pl.ckpt
Moved: =4.27.7
Moved: Step4.ipynb
Moved: custom_main.py
Moved: __pycache__
Moved: coco-classes-mapping-master
Moved: generate_mapping.py
Moved: coco80.names
Moved: coco91.names
Moved: coco_mapping_80to91.json
Moved: coco_mapping_91to80.json
Moved: zero_shot_eval.py
Moved: Step4_Professional_Cleaned.ipynb
Extraction complete!


## 1. Zero-Redundancy Model Loader
We avoid manual layer construction. This function uses the `class_path` from the YAMLs to dynamicallly instantiate the correct classes.

In [ ]:
def load_project_model(config_path, checkpoint_path):
    with open(config_path, "r") as f: cfg = yaml.safe_load(f)

    # A. Setup Data (Extract metadata like img_size)
    d_mod, d_cls = cfg["data"]["class_path"].rsplit(".", 1)
    dm = getattr(importlib.import_module(d_mod), d_cls)(path="./data", batch_size=1, num_workers=0, **cfg["data"].get("init_args", {}))

    # B. Setup Model (Recursive build of Encoder -> Network -> LightningModule)
    m_mod, m_cls = cfg["model"]["class_path"].rsplit(".", 1)
    net_args = cfg["model"]["init_args"]["network"]

    from models.vit import ViT
    from models.eomt import EoMT

    encoder = ViT(img_size=dm.img_size, **net_args["init_args"]["encoder"]["init_args"])
    network = EoMT(encoder=encoder, num_classes=dm.num_classes, **{k:v for k,v in net_args["init_args"].items() if k != "encoder"})

    model = getattr(importlib.import_module(m_mod), m_cls)(
        network=network, img_size=dm.img_size, num_classes=dm.num_classes,
        **{k:v for k,v in cfg["model"]["init_args"].items() if k != "network"}
    )

    # C. State Dict Injection
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
    return model.to(device).eval(), dm

model_cs, dm_cs = load_project_model("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "eomt_cityscapes_pl.ckpt")
model_coco, _ = load_project_model("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "eomt_coco.bin")

## 2. Visualization (High-Level Methods)
We use the framework's internal `window_imgs_semantic` and `resize_and_pad_imgs_instance_panoptic` to visualize the same image through two different task lenses.

In [ ]:
dm_cs.setup(); img, _ = dm_cs.cityscapes_val_dataset[0]
orig_size = img.shape[-2:]

with torch.no_grad():
    # 1. Cityscapes Semantic (Sliding Window)
    crops, origins = model_cs.window_imgs_semantic([img])
    m_l, c_l = model_cs(crops.to(device))
    m_l = F.interpolate(m_l[-1], model_cs.img_size, mode="bilinear")
    crop_logits = model_cs.to_per_pixel_logits_semantic(m_l, c_l[-1])
    pred_cs = model_cs.revert_window_logits_semantic(crop_logits, origins, [orig_size])[0].argmax(0)

    # 2. COCO Panoptic (Padded Resize)
    tx = model_coco.resize_and_pad_imgs_instance_panoptic([img.to(device)])
    mp, cp = model_coco(tx)
    mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"), [orig_size])
    pred_coco = model_coco.to_per_pixel_preds_panoptic(mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8)[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img.permute(1, 2, 0).numpy()); axes[0].set_title("Input Image")
axes[1].imshow(pred_cs.cpu().numpy()); axes[1].set_title("Semantic Map (Classes)")
axes[2].imshow(pred_coco[..., 1].cpu().numpy()); axes[2].set_title("Panoptic Map (Instances)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 3. The Semantic Bridge (Quantitative Mapping)
We consolidate the `bridge` and `stuff_bridge` into a single translation function. We evaluate on 19 classes, treating everything else as class ID 19 (Ignore).

In [ ]:
# Load the index translation (0-79 model index -> 1-91 COCO ID)
import json
map_file_path = os.path.join(project_root, 'coco-classes-mapping-master/coco_mapping_80to91.json')
with open(map_file_path, 'r') as f: coco_idx_map = {int(k)-1: int(v) for k, v in json.load(f).items()}

# Build the final bridge (COCO IDs -> Cityscapes Train IDs)
things_map = { 1: 11, 2: 18, 3: 13, 4: 17, 6: 15, 7: 16, 8: 14, 10: 6, 13: 7 }
stuff_map = { 100: 0, 123: 1, 91: 2, 129: 2, 109: 3, 110: 3, 111: 3, 112: 3, 131: 3, 117: 4, 116: 8, 125: 8, 126: 9, 119: 10 }

def bridge_to_cs(pred_tensor):
    res = torch.full_like(pred_tensor, 19)
    for idx, coco_id in coco_idx_map.items():
        if coco_id in things_map: res[pred_tensor == idx] = things_map[coco_id]
    for stuff_id, cs_id in stuff_map.items():
        res[pred_tensor == stuff_id] = cs_id
    return res

evaluator = iouEval(20) # 19 classes + 1 ignore bucket
for batch in tqdm(dm_cs.val_dataloader(), desc="Evaluating Mapped COCO"):
    imgs, targets = batch
    gt = model_cs.to_per_pixel_targets_semantic(targets, 19)[0].to(device) # High-level framework GT conversion

    with torch.no_grad():
        # Predict using COCO model
        tx = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"), [imgs[0].shape[-2:]])
        pred = model_coco.to_per_pixel_preds_panoptic(mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8)[0][..., 0]

        # Bridge and Eval
        evaluator.addBatch(bridge_to_cs(pred).unsqueeze(0).unsqueeze(0), gt.unsqueeze(0).unsqueeze(0))

_, ious = evaluator.getIoU()
print(f"\nZero-Shot COCO mIoU on Cityscapes: {ious[:19].mean()*100:.2f}%")

### 🚀 Safely Push to Remote (Git)

1. **Configure Git:** Sets your name and email (if not already set in this Colab session).
2. **Stage & Commit:** Adds all the moved files and commits them.
3. **Pull with Rebase:** `git pull --rebase origin main` fetches any new changes from the remote and replays your local commits on top of them. This avoids merge conflict commits and keeps the history linear.
4. **Push:** Sends the changes to GitHub.

*Note: If you get an authentication error, you will need to update your remote URL to include a GitHub Personal Access Token:*
`!git remote set-url origin https://<YOUR_TOKEN>@github.com/<USERNAME>/<REPO>.git`

In [9]:
import os
import subprocess
from google.colab import userdata

# 1. Fetch credentials from Colab Secrets
try:
    git_user = userdata.get('gituser')
    # Make sure to add your Personal Access Token as 'GITHUB_TOKEN' in Secrets (ቑ1)!
    git_token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError as e:
    print(f"Error: {e}")
    print("Please ensure both 'gituser' and 'GITHUB_TOKEN' are added to the Colab Secrets.")
    raise

# 2. Configuration
git_email = "s360426@studenti.polito.it"  # Updated to your email
repo_name = "Fundamental_Project"      # Updated to match your repo name

# 3. Setup Identity and Remote URL
os.chdir('/content/drive/MyDrive/FundGitHubProject')
os.system(f'git config --global user.name "{git_user}"')
os.system(f'git config --global user.email "{git_email}"')

# Securely set the remote URL with the token
remote_url = f"https://{git_token}@github.com/{git_user}/{repo_name}.git"
os.system(f'git remote set-url origin {remote_url}')

# Remove stalled git lock file if it exists
lock_file = '.git/index.lock'
if os.path.exists(lock_file):
    os.remove(lock_file)
    print(f"Removed stalled git lock file: {lock_file}")

# 4. Run Git Commands
git_commands = [
    "git add .",
    'git commit -m "Update .gitignore and reorganize weights"',
    "git fetch --all",
    "git pull --rebase origin main",
    "git push origin --all" # Pushes all local branches to the remote
]

for cmd in git_commands:
    print(f"\nExecuting: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())

print("\n✅ Git operations sequence completed.")


Executing: git add .

Executing: git commit -m "Refactor: Extract eomt subfolders to project root"
[matteo_branch 1323344] Refactor: Extract eomt subfolders to project root
 78 files changed, 423 insertions(+), 174 deletions(-)
 rename eomt/.gitignore => .gitignore (96%)
 rename eomt/=4.27.7 => =4.27.7 (100%)
 rename eomt/LICENSE => LICENSE (100%)
 rename eomt/README.md => README.md (100%)
 create mode 100644 Step4.ipynb
 create mode 100644 Step4_Professional_Cleaned.ipynb
 rename eomt/__init__.py => __init__.py (100%)
 rename {eomt/coco-classes-mapping => coco-classes-mapping-master}/README.md (100%)
 rename {eomt/coco-classes-mapping => coco-classes-mapping-master}/coco80.names (100%)
 rename {eomt/coco-classes-mapping => coco-classes-mapping-master}/coco91.names (100%)
 rename {eomt/coco-classes-mapping => coco-classes-mapping-master}/coco_mapping_80to91.json (100%)
 rename {eomt/coco-classes-mapping => coco-classes-mapping-master}/coco_mapping_91to80.json (100%)
 rename {eomt/coco

In [8]:
import os

# Go to the project directory
os.chdir('/content/drive/MyDrive/FundGitHubProject')

# Items to ignore so git add doesn't stall
ignore_list = [
    "*.bin",
    "*.ckpt",
    "data/",
    "datasets/",
    "__pycache__/",
    ".ipynb_checkpoints/"
]

# Append them to .gitignore
with open('.gitignore', 'a') as f:
    f.write('\n# Automatically ignored large files to prevent Git stalling\n')
    for item in ignore_list:
        f.write(f'{item}\n')

print("✅ Added large files and directories to .gitignore.")
print("You can now run the Git Push cell again! It should be much faster.")

✅ Added large files and directories to .gitignore.
You can now run the Git Push cell again! It should be much faster.


In [10]:
%pwd

'/content/drive/MyDrive/FundGitHubProject'

In [14]:
!mv eomt_* eomt/eomt_weights/